# 03 · Modelos base

Validación y primeros resultados. El test no se toca: todo con validación cruzada dentro de train.

In [1]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

from src.preprocesamiento import COLUMNAS, crear_preprocesador

train = pd.read_csv('../data/processed/train.csv')
X_train = train[COLUMNAS]
y_train = train['FraudFound_P']
print(X_train.shape, y_train.sum(), 'fraudes')

(12335, 19) 738 fraudes


## 1. Por qué validación cruzada

Mismo modelo en 20 particiones 80/20 distintas.

In [2]:
modelo = make_pipeline(crear_preprocesador(), HistGradientBoostingClassifier(random_state=42))

particiones = StratifiedShuffleSplit(n_splits=20, test_size=0.2, random_state=0)
resultado = cross_validate(modelo, X_train, y_train, cv=particiones, scoring='average_precision')['test_score']

print('PR-AUC en 20 particiones distintas')
print('media:', resultado.mean().round(3), '| desviación:', resultado.std().round(3))
print('mínimo:', resultado.min().round(3), '| máximo:', resultado.max().round(3))

PR-AUC en 20 particiones distintas
media: 0.214 | desviación: 0.025
mínimo: 0.168 | máximo: 0.264


El mismo modelo da resultados distintos según la partición (desviación de 0,025): diferencias menores que esa no se distinguen del azar.

Decisión: validación cruzada estratificada de 5 particiones, con el preprocesador dentro del Pipeline.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metricas = {'PR-AUC': 'average_precision', 'ROC-AUC': 'roc_auc'}

## 2. Baselines

Dummy, regresión logística e HistGradientBoosting.

In [4]:
modelos = {
    'Dummy': make_pipeline(crear_preprocesador(), DummyClassifier(strategy='prior')),
    'Regresión logística': make_pipeline(crear_preprocesador(), SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000)),
    'HistGradientBoosting': make_pipeline(crear_preprocesador(), HistGradientBoostingClassifier(random_state=42)),
}

filas = []
for nombre, m in modelos.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring=metricas)
    filas.append({'modelo': nombre,
                  'PR-AUC': r['test_PR-AUC'].mean(), 'PR-AUC std': r['test_PR-AUC'].std(),
                  'ROC-AUC': r['test_ROC-AUC'].mean(), 'ROC-AUC std': r['test_ROC-AUC'].std()})

pd.DataFrame(filas).set_index('modelo').round(3)

,PR-AUC,PR-AUC std,ROC-AUC,ROC-AUC std
modelo,,,,
Dummy,0.060,0.000,0.500,0.000
Regresión logística,0.167,0.025,0.797,0.007
HistGradientBoosting,0.224,0.031,0.811,0.009


Los dos modelos superan el suelo (0,06). HistGradientBoosting (0,224) supera a la regresión logística (0,167) por más que el ruido; en ROC-AUC casi no se distinguen (0,811 frente a 0,797).

## 3. Qué variables sostienen el modelo (ablación)

Se quita cada variable y se mide cuánto baja la PR-AUC de HistGradientBoosting (validación cruzada 5 × 3, mismas particiones).

In [5]:
from sklearn.model_selection import RepeatedStratifiedKFold

cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

def pr_auc(columnas):
    m = make_pipeline(crear_preprocesador(columnas), HistGradientBoostingClassifier(random_state=42))
    return cross_validate(m, X_train[columnas], y_train, cv=cv_rep, scoring='average_precision')['test_score'].mean()

base = pr_auc(COLUMNAS)
print('PR-AUC con todas las variables:', round(base, 3))

PR-AUC con todas las variables: 0.222


In [6]:
sin_variable = {c: pr_auc([x for x in COLUMNAS if x != c]) for c in COLUMNAS}
ablacion = pd.Series(sin_variable).sub(base).sort_values()
ablacion.round(3).rename('cambio en PR-AUC al quitarla')

Fault                  -0.082
BasePolicy             -0.020
AgentType              -0.002
AccidentArea           -0.001
AgeOfVehicle           -0.001
retraso_meses          -0.001
MonthClaimed           -0.001
Deductible             -0.000
VehicleCategory        -0.000
Make                   -0.000
PastNumberOfClaims     -0.000
Sex                    -0.000
AgeOfPolicyHolder      -0.000
Month                   0.000
VehiclePrice            0.001
Days_Policy_Accident    0.001
Age                     0.001
AddressChange_Claim     0.001
Days_Policy_Claim       0.003
Name: cambio en PR-AUC al quitarla, dtype: float64

Ablación por grupos de variables que se solapan:

In [7]:
grupos = {
    'fechas (Month, MonthClaimed, retraso_meses)': ['Month', 'MonthClaimed', 'retraso_meses'],
    'edades (Age, AgeOfPolicyHolder)': ['Age', 'AgeOfPolicyHolder'],
    'póliza (BasePolicy, VehicleCategory)': ['BasePolicy', 'VehicleCategory'],
    'vehículo (Make, VehiclePrice, AgeOfVehicle)': ['Make', 'VehiclePrice', 'AgeOfVehicle'],
    'días de póliza (Days_Policy_Accident, Days_Policy_Claim)': ['Days_Policy_Accident', 'Days_Policy_Claim'],
}
pd.Series({g: pr_auc([c for c in COLUMNAS if c not in cols]) - base for g, cols in grupos.items()}).round(3).rename('cambio en PR-AUC al quitar el grupo')

fechas (Month, MonthClaimed, retraso_meses)                -0.003
edades (Age, AgeOfPolicyHolder)                            -0.018
póliza (BasePolicy, VehicleCategory)                       -0.052
vehículo (Make, VehiclePrice, AgeOfVehicle)                -0.006
días de póliza (Days_Policy_Accident, Days_Policy_Claim)    0.001
Name: cambio en PR-AUC al quitar el grupo, dtype: float64

Fault sostiene el modelo: quitarla baja la PR-AUC de 0,222 a 0,14 (−37 %). La póliza (BasePolicy + VehicleCategory) es la segunda fuente de señal (−0,052) y las edades aportan algo (−0,018). El resto está por debajo del ruido (cambios de 0,003 o menos).

No se sabe cuándo se registra Fault.

Decisión: entrenar y reportar dos versiones, con Fault (techo de rendimiento) y sin Fault (conservadora).

## 4. Desbalance: class_weight

Más peso a los fraudes, con y sin Fault.

In [8]:
sin_fault = [c for c in COLUMNAS if c != 'Fault']

def modelo(nombre, columnas, peso):
    if nombre == 'Regresión logística':
        return make_pipeline(crear_preprocesador(columnas), SimpleImputer(strategy='median'), StandardScaler(),
                             LogisticRegression(max_iter=1000, class_weight=peso))
    return make_pipeline(crear_preprocesador(columnas), HistGradientBoostingClassifier(class_weight=peso, random_state=42))

filas = []
for version, columnas in {'con Fault': COLUMNAS, 'sin Fault': sin_fault}.items():
    for nombre in ['Regresión logística', 'HistGradientBoosting']:
        for peso in [None, 'balanced']:
            r = cross_validate(modelo(nombre, columnas, peso), X_train[columnas], y_train, cv=cv_rep, scoring='average_precision')['test_score']
            filas.append({'versión': version, 'modelo': nombre, 'class_weight': peso, 'PR-AUC': r.mean(), 'std': r.std()})

pd.DataFrame(filas).round(3)

,versión,modelo,class_weight,PR-AUC,std
0,con Fault,Regresión logística,NaN,0.169,0.017
1,con Fault,Regresión logística,balanced,0.160,0.014
2,con Fault,HistGradientBoosting,NaN,0.222,0.029
3,con Fault,HistGradientBoosting,balanced,0.217,0.031
4,sin Fault,Regresión logística,NaN,0.145,0.025
5,sin Fault,Regresión logística,balanced,0.143,0.024
6,sin Fault,HistGradientBoosting,NaN,0.140,0.018
7,sin Fault,HistGradientBoosting,balanced,0.139,0.021


class_weight no mejora la PR-AUC (diferencias de 0,002 a 0,009, dentro del ruido): esta métrica depende del orden de las probabilidades, no de su escala. Con Fault, HistGradientBoosting (0,222) supera a la regresión logística (0,169); sin Fault empatan (0,140 y 0,145), así que su ventaja venía de la interacción de Fault con el resto.

Decisión: no usar class_weight ni SMOTE.

## 5. Más modelos

RandomForest, XGBoost y LightGBM con valores por defecto, con y sin Fault.

In [9]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

def pipeline(clasificador, columnas):
    return make_pipeline(crear_preprocesador(columnas), clasificador)

clasificadores = {
    'Regresión logística': lambda c: make_pipeline(crear_preprocesador(c), SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000)),
    'HistGradientBoosting': lambda c: pipeline(HistGradientBoostingClassifier(random_state=42), c),
    'RandomForest': lambda c: make_pipeline(crear_preprocesador(c), SimpleImputer(strategy='median'), RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)),
    'XGBoost': lambda c: pipeline(XGBClassifier(random_state=42, n_jobs=-1), c),
    'LightGBM': lambda c: pipeline(LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1), c),
}

filas = []
for version, columnas in {'con Fault': COLUMNAS, 'sin Fault': sin_fault}.items():
    for nombre, crear in clasificadores.items():
        r = cross_validate(crear(columnas), X_train[columnas], y_train, cv=cv_rep, scoring='average_precision')['test_score']
        filas.append({'versión': version, 'modelo': nombre, 'PR-AUC': r.mean(), 'std': r.std()})

comparacion = pd.DataFrame(filas)
comparacion.pivot(index='modelo', columns='versión', values='PR-AUC').round(3)

versión,con Fault,sin Fault
modelo,,
HistGradientBoosting,0.222,0.140
LightGBM,0.222,0.142
RandomForest,0.193,0.111
Regresión logística,0.169,0.145
XGBoost,0.217,0.132


Con Fault, HistGradientBoosting y LightGBM empatan (0,222), XGBoost casi (0,217) y RandomForest queda por debajo (0,193). Sin Fault gana la regresión logística (0,145); XGBoost (0,132) y RandomForest (0,111) sobreajustan.

Decisión: ajustar LightGBM y la regresión logística; descartar RandomForest y XGBoost.